In [1]:
import pandas as pd, torch
from unfairness.utils.config_loader import load_model_and_tokenizer
from unfairness.utils.constant import KB_CATEGORIES
from unfairness.dataset import ToS, make_dataloaders
from unfairness.infer import predict_with_rationales

CKPT     = "cv_test/torch/fold_1/best-v7.ckpt"     
KB_DIR   = "local_database/KB"
TEST_CSV = "cv_test/torch/fold_1/test.csv"      
MAX_LEN  = 128
BATCH    = 64
THRESH   = 0.5          
TOP_K    = 3
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"

In [2]:
lit, tok, kb_struct = load_model_and_tokenizer(
    ckpt_path=CKPT,
    max_len=MAX_LEN,
    kb_dir=KB_DIR,
    map_location=DEVICE,
)
lit.eval()
print("device:", next(lit.parameters()).device, "vocab:", tok.vocab_size)

device: cuda:0 vocab: 9817


In [5]:
df_test = pd.read_csv(TEST_CSV)
ds_test = ToS(df_test, tok, MAX_LEN)          
_, _, te_loader = make_dataloaders(ds_test, None, ds_test, batch_size=BATCH)
len(df_test), next(iter(te_loader))["input_ids"].shape

results = predict_with_rationales(
    lit_module=lit,
    dataloader=te_loader,
    kb_struct_or_texts=kb_struct,   
    threshold=THRESH,
    top_k=TOP_K,          
    use_scores=True,
    device = DEVICE
)

# Muestra 3 filas
for r in results[:3]:
    print("TEXT:", r["text"][:180].replace("\n"," "), "...")
    for cat in KB_CATEGORIES:
        pc = r["per_category"][cat]
        if pc["pred"] == 1 and pc["rationales"]:
            top = pc["rationales"][0]
            print(f" * {cat}: prob={pc['prob']:.3f} -> ({top['idx']}, {top['tag']}) {top['text'][:90]}...")
    print("-"*80)
len(results)

TEXT: your agreement to this tos constitutes your agreement that you are deemed to have received any and all notices that would have been delivered had you accessed the yahoo services in ...
--------------------------------------------------------------------------------
TEXT: you must sign the opt-out notice for it to be effective . ...
--------------------------------------------------------------------------------
TEXT: some states do not allow the exclusion or limitation of incidental or consequential damages , so the above limitation or exclusion may not apply to you , in which case such exclusi ...
 * LTD: prob=0.612 -> (21, ltd3) Liability is excluded also in cases of physical or personal injuries....
--------------------------------------------------------------------------------


3063

In [8]:
import json

with open('infer.json', 'w') as f:
    json.dump(results, f)